# 02 - Data Cleaning and Preprocessing

**Project:** Pump It Up — Data Mining the Water Table
**Author:** Jarret Angbazo | Date: June 2026

---

### Objective
Produce clean train/test sets ready for feature engineering.
 - Handle explicit nulls and implicit zeros
 - Drop leaky / zero-variance / near-duplicate columns
 - Standardise data types
 - Encode target


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path('../')
RAW = ROOT / 'data/raw'
PROCESSED = ROOT / 'data/processed'
PROCESSED.mkdir(parents=True, exist_ok=True)


---------------------------------------------------------------------------
1. Load
---------------------------------------------------------------------------

In [14]:
feat   = pd.read_csv(RAW / "training_set_features.csv")
labels = pd.read_csv(RAW / "training_set_labels.csv")
test   = pd.read_csv(RAW / "test_set_features.csv")
df     = feat.merge(labels, on="id").copy()
print(f"Raw train shape: {df.shape}")


Raw train shape: (59400, 41)


---------------------------------------------------------------------------
2. Drop columns
Rationale documented for each decision
---------------------------------------------------------------------------

In [15]:
drop_cols = [
    "recorded_by",        # Single value — zero variance
    "wpt_name",           # 37k unique pump names — not generalisable
    "subvillage",         # 19k unique — use region/lga/ward hierarchy instead
    "scheme_name",        # 48.5% missing AND 2695 unique — too sparse
    "num_private",        # Near-constant (>95% zero)
    "quantity_group",     # Exact duplicate of 'quantity'
    "payment_type",       # Exact duplicate of 'payment'
    "source_type",        # Collapsed by 'source_class' (3 levels vs 7)
    "extraction_type_group",  # Redundant with extraction_type / class hierarchy
    "management_group",   # Collapsed from 'management' (5 vs 12 levels)
    "quality_group",      # Redundant with 'water_quality'
    "waterpoint_type_group",  # Redundant with 'waterpoint_type'
]
df   = df.drop(columns=drop_cols, errors="ignore")
test = test.drop(columns=drop_cols, errors="ignore")
print(f"After dropping redundant cols: {df.shape}")


After dropping redundant cols: (59400, 29)


---------------------------------------------------------------------------
3. Date feature — parse and extract year/month/age
---------------------------------------------------------------------------

In [16]:
for frame in [df, test]:
    frame["date_recorded"] = pd.to_datetime(frame["date_recorded"])
    frame["recorded_year"]  = frame["date_recorded"].dt.year
    frame["recorded_month"] = frame["date_recorded"].dt.month
df   = df.drop(columns=["date_recorded"])
test = test.drop(columns=["date_recorded"])


---------------------------------------------------------------------------
4. Implicit zeros → NaN
(0 in these columns indicates missing, not a real measurement)
---------------------------------------------------------------------------

In [17]:
zero_to_nan = ["construction_year", "longitude", "population", "gps_height"]
for col in zero_to_nan:
    df.loc[df[col] == 0, col]     = np.nan
    test.loc[test[col] == 0, col] = np.nan

# amount_tsh: 70% zeros — keep as-is but add missingness indicator
df["amount_tsh_missing"]   = (df["amount_tsh"] == 0).astype(int)
test["amount_tsh_missing"] = (test["amount_tsh"] == 0).astype(int)
df.loc[df["amount_tsh"] == 0, "amount_tsh"]     = np.nan
test.loc[test["amount_tsh"] == 0, "amount_tsh"] = np.nan


---------------------------------------------------------------------------
5. Pump age feature (requires construction_year; compute before imputing)
---------------------------------------------------------------------------

In [18]:
for frame in [df, test]:
    frame["pump_age"] = frame["recorded_year"] - frame["construction_year"]
    # Negative or implausible ages → NaN
    frame.loc[frame["pump_age"] < 0, "pump_age"] = np.nan
    frame.loc[frame["pump_age"] > 80, "pump_age"] = np.nan


---------------------------------------------------------------------------
6. Imputation
Numeric → median (robust to skew)
Categorical → mode / "Unknown"
All imputers fit on TRAIN only, applied to both
---------------------------------------------------------------------------

6a. Numeric median imputation (fit on train)

In [19]:
num_impute_cols = ["construction_year", "longitude", "latitude",
                   "gps_height", "population", "amount_tsh", "pump_age"]
medians = {}
for col in num_impute_cols:
    if col in df.columns:
        med = df[col].median()
        medians[col] = med
        df[col]   = df[col].fillna(med)
        test[col] = test[col].fillna(med)

# 6b. Boolean columns: public_meeting, permit — treat as strings, fill "Unknown"
bool_cols = ["public_meeting", "permit"]
for col in bool_cols:
    for frame in [df, test]:
        frame[col] = frame[col].astype(str).replace("nan", "Unknown")

# 6c. Remaining categorical nulls → "Unknown"
cat_cols_with_na = ["funder", "installer", "scheme_management", "lga", "ward"]
for col in cat_cols_with_na:
    for frame in [df, test]:
        if col in frame.columns:
            frame[col] = frame[col].fillna("Unknown")


---------------------------------------------------------------------------
7. High-cardinality collapse: funder, installer
Keep top-N categories; lump the rest as "Other"
---------------------------------------------------------------------------

In [20]:
TOP_N = 20
for col in ["funder", "installer"]:
    top_cats = df[col].value_counts().nlargest(TOP_N).index
    df[col]   = df[col].where(df[col].isin(top_cats), "Other")
    test[col] = test[col].where(test[col].isin(top_cats), "Other")
    print(f"  {col}: kept {TOP_N} top categories + 'Other'")


  funder: kept 20 top categories + 'Other'
  installer: kept 20 top categories + 'Other'


---------------------------------------------------------------------------
8. Encode target (keep original string label too for readability)
---------------------------------------------------------------------------

In [21]:
label_map = {"functional": 2, "functional needs repair": 1, "non functional": 0}
df["status_group_enc"] = df["status_group"].map(label_map)


---------------------------------------------------------------------------
9. Final check
---------------------------------------------------------------------------

In [22]:
print(f"\nCleaned train shape  : {df.shape}")
print(f"Cleaned test shape   : {test.shape}")
null_train = df.drop(columns=["status_group","status_group_enc"]).isnull().sum()
remaining = null_train[null_train > 0]
if len(remaining) == 0:
    print("No remaining nulls in train ✓")
else:
    print("Remaining nulls:\n", remaining)

print("\nData types:")
print(df.dtypes.value_counts())



Cleaned train shape  : (59400, 33)
Cleaned test shape   : (14850, 31)
No remaining nulls in train ✓

Data types:
object     19
float64     7
int64       5
int32       2
Name: count, dtype: int64


---------------------------------------------------------------------------
10. Save
---------------------------------------------------------------------------

In [23]:
df.to_csv(PROCESSED / "train_cleaned.csv", index=False)
test.to_csv(PROCESSED / "test_cleaned.csv", index=False)
pd.Series(medians).to_csv(PROCESSED / "train_medians.csv")
print(f"\nSaved: train_cleaned.csv, test_cleaned.csv, train_medians.csv")

# Summary
print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)
print(f"  Columns dropped       : {len(drop_cols)}")
print(f"  Features remaining    : {df.shape[1] - 2}")  # minus 2 target cols
print(f"  Implicit zeros fixed  : construction_year, longitude, gps_height, population, amount_tsh")
print(f"  New features added    : pump_age, amount_tsh_missing, recorded_year, recorded_month")
print(f"  High-card collapsed   : funder ({TOP_N} cats), installer ({TOP_N} cats)")
print(f"  Train medians stored  : will be applied to test set downstream")
# (fix applied inline above - re-run fix)



Saved: train_cleaned.csv, test_cleaned.csv, train_medians.csv

CLEANING SUMMARY
  Columns dropped       : 12
  Features remaining    : 31
  Implicit zeros fixed  : construction_year, longitude, gps_height, population, amount_tsh
  New features added    : pump_age, amount_tsh_missing, recorded_year, recorded_month
  High-card collapsed   : funder (20 cats), installer (20 cats)
  Train medians stored  : will be applied to test set downstream
